# Burunov Bot — тест F5-TTS-Russian на готовом датасете

**Датасет:** https://github.com/shray77/burunov-voice
- 840 сегментов (110+ минут чистого голоса Бурунова)
- 22050 Hz, mono, WAV PCM 16-bit
- 315 сегментов с LLM-коррекцией транскрипции
- 265 сегментов в диапазоне 5-15 сек (идеально для референса F5-TTS)

**Цель ноутбука:**
1. Загрузить датасет
2. Найти лучший референс (5-15 сек, чёткий текст)
3. Протестировать F5-TTS-Russian (zero-shot клон)
4. Замерить RTF (real-time factor) на T4
5. Сгенерировать пресеты анекдотов для демо
6. Скачать модели и пресеты для переноса на G1

**GPU:** Colab T4 (бесплатно, ~12ч/день)

## 1. Установка F5-TTS (3-5 минут)

In [ ]:
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
%cd /content/F5-TTS
!pip install -e . 2>&1 | tail -5
!pip install huggingface_hub soundfile torchaudio

## 2. Проверка GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU НЕ доступно! Runtime → Change runtime type → T4 GPU')

## 3. Загрузка датасета burunov-voice

In [ ]:
!git clone https://github.com/shray77/burunov-voice.git /content/burunov-voice 2>&1 | tail -3

import os, json

DATASET_DIR = '/content/burunov-voice'
SEGMENTS_DIR = os.path.join(DATASET_DIR, 'segments')
MANIFEST_PATH = os.path.join(DATASET_DIR, 'manifest.json')

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print(f'✅ Датасет загружен')
print(f'   Сегментов: {len(manifest)}')
print(f'   С LLM-коррекцией: {sum(1 for m in manifest if m.get("text_corrected"))}')
print(f'   Общая длительность: {sum(m["duration"] for m in manifest)/60:.1f} мин')

## 4. Найти лучшие референсы для F5-TTS

Критерии идеального референса:
- 5-15 секунд (F5-TTS любит этот диапазон)
- Есть LLM-коррекция (текст более точный)
- Текст состоит из законченных предложений
- Достаточно длинный (не одно слово)

In [ ]:
# Выбираем кандидатов
candidates = []
for m in manifest:
    dur = m['duration']
    text = m.get('text_corrected') or m.get('text', '')
    if not text:
        continue
    # 5-15 сек, есть скорректированный текст, длина текста адекватная
    if 5.0 <= dur <= 15.0 and m.get('text_corrected') and len(text) > 30:
        # Подсчёт "качества" — чем больше слов и яснее текст, тем лучше
        words = text.split()
        if len(words) >= 5:
            candidates.append({
                'audio_path': os.path.join(DATASET_DIR, m['audio_path']),
                'text': text,
                'duration': dur,
                'word_count': len(words),
                'source': m['source'],
            })

# Сортируем по длительности (близко к 10 сек — оптимально)
candidates.sort(key=lambda c: abs(c['duration'] - 10))

print(f'Найдено {len(candidates)} кандидатов в референсы')
print(f'\nТоп-10 кандидатов (ближайшие к 10 сек):')
for i, c in enumerate(candidates[:10]):
    print(f'\n[{i}] {c["duration"]:.1f}сек, {c["word_count"]} слов')
    print(f'    Текст: {c["text"][:120]}')
    print(f'    Файл: {os.path.basename(c["audio_path"])}')

## 5. Послушать кандидатов (выбрать вручную)

Запусти ячейку, прослушай 5 кандидатов, выбери тот где голос самый чистый и характерный.
Запиши номер в `CHOSEN_REF_INDEX` ниже.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

for i, c in enumerate(candidates[:5]):
    print(f'\n>>> [{i}] {c["duration"]:.1f}сек')
    print(f'    {c["text"]}')
    audio, sr = sf.read(c['audio_path'])
    print(f'    SR={sr}, channels={audio.shape}')
    display(Audio(c['audio_path']))

## 6. Установить выбранный референс

После прослушивания поменяй `CHOSEN_REF_INDEX` на номер лучшего кандидата (0-4).

In [ ]:
# ⚠️ ПОМЕНЯЙ НОМЕР после прослушивания выше
CHOSEN_REF_INDEX = 0

ref = candidates[CHOSEN_REF_INDEX]
REF_AUDIO = ref['audio_path']
REF_TEXT = ref['text']

print(f'✅ Выбран референс #{CHOSEN_REF_INDEX}')
print(f'   Файл: {REF_AUDIO}')
print(f'   Длительность: {ref["duration"]:.1f}сек')
print(f'   Текст: "{REF_TEXT}"')

# Копируем в стандартное место для дальнейшего использования
import shutil
shutil.copy(REF_AUDIO, '/content/burunov_ref.wav')
with open('/content/burunov_ref.txt', 'w') as f:
    f.write(REF_TEXT)
print(f'\nСкопировано в /content/burunov_ref.wav + .txt')

## 7. Скачать модель hotstone228/F5-TTS-Russian

In [ ]:
from huggingface_hub import snapshot_download

print('Скачиваем модель F5-TTS-Russian...')
model_path = snapshot_download(repo_id='hotstone228/F5-TTS-Russian')
print(f'✅ Модель скачана: {model_path}')

print('\nФайлы в репо модели:')
for f in os.listdir(model_path):
    full = os.path.join(model_path, f)
    if os.path.isfile(full):
        size = os.path.getsize(full) / 1e6
        print(f'  {f} ({size:.1f} MB)')

## 8. Загрузка F5-TTS

In [ ]:
import sys
sys.path.insert(0, '/content/F5-TTS')

from f5_tts.api import F5TTS

print('Инициализация F5TTS...')
tts = F5TTS(
    model_type='F5-TTS',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print('✅ F5-TTS готова')

## 9. Тест синтеза + замер RTF

RTF (real-time factor) = время_синтеза / длительность_аудио
- RTF < 1.0 = real-time, можно live
- RTF 1-3 = терпимо с задержкой
- RTF > 3 = плохо, лучше Piper

In [ ]:
import time
import soundfile as sf
import numpy as np

# Тестовые фразы из сценария Бурунова
TEST_TEXTS = [
    'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'Бля, тут кто-то стоит... дай пройти.',
    'Вот ваш кофе, Олег. Не обожгись, бля.',
    'Учительница спрашивает Вовочку: Вовочка, какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
]

results = []
os.makedirs('/content/test_outputs', exist_ok=True)

for i, text in enumerate(TEST_TEXTS):
    print(f'\n[{i+1}/{len(TEST_TEXTS)}] Синтез: "{text[:60]}..."')
    out_path = f'/content/test_outputs/test_{i}.wav'
    
    t0 = time.time()
    try:
        wav, sr, _ = tts.infer(
            ref_file=REF_AUDIO,
            ref_text=REF_TEXT,
            gen_text=text,
            file_wave=out_path,
        )
        synth_time = time.time() - t0
        
        audio, sr_out = sf.read(out_path)
        audio_dur = len(audio) / sr_out
        rtf = synth_time / audio_dur
        
        print(f'  Синтез: {synth_time:.2f}с | Аудио: {audio_dur:.2f}с | RTF={rtf:.2f}')
        results.append({'text': text, 'synth': synth_time, 'audio': audio_dur, 'rtf': rtf, 'path': out_path})
        
        if rtf > 3:
            print('  ⚠️ RTF > 3 — на демо будет запаздывать')
    except Exception as e:
        print(f'  ❌ Ошибка: {e}')
        import traceback; traceback.print_exc()

## 10. Сводка RTF + прослушивание

In [ ]:
if results:
    print('\n=== СВОДКА RTF ===')
    print(f'{"Текст":<50} {"Синтез":<10} {"Аудио":<10} {"RTF":<8}')
    print('-' * 80)
    for r in results:
        text_short = r['text'][:48]
        print(f'{text_short:<50} {r["synth"]:<10.2f} {r["audio"]:<10.2f} {r["rtf"]:<8.2f}')
    avg_rtf = sum(r['rtf'] for r in results) / len(results)
    print(f'\nСредний RTF: {avg_rtf:.2f}')
    if avg_rtf < 1.0:
        print('✅ ОТЛИЧНО — real-time, можно использовать на G1')
    elif avg_rtf < 2.0:
        print('🟡 ТЕРПИМО — будет небольшая задержка (1-2сек)')
    else:
        print('🔴 ПЛОХО — переключаемся на Piper ONNX')

    print('\n=== Прослушать ===')
    for r in results:
        print(f'\n>>> {r["text"]}')
        display(Audio(r['path']))

## 11. Сгенерировать пресеты анекдотов для демо

10 файлов: 5 тем × (intro + joke). Эти wav-файлы потом зальём на G1 как fallback если RAG/LLM упадёт.

In [ ]:
PRESETS = {
    # Штирлиц
    'shtirlits_intro': 'Ну, значицца, Штирлиц... он вообще почти весь восемьдесят шестой по телевизору был, ну вы помните.',
    'shtirlits_joke': 'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    # Вовочка
    'volodka_intro': 'А вот Вовочка... ну, который из школы... этот, как его... хулиган.',
    'volodka_joke': 'Учительница спрашивает Вовочку: Вовочка, какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
    # Поручик Ржевский
    'rzhevsky_intro': 'Ну этот... поручик наш... Ржевский, да... он вообще к дамам постоянно лез.',
    'rzhevsky_joke': 'Поручик Ржевский на балу подходит к Наташе Ростовой: Наташа, а Наташа, давайте я вас поцелую! Поручик, я же дама! Ну, тем более!',
    # Новые русские
    'new_russians_intro': 'А это ещё тогда, в восемьдесят шестом, кооперативы пошли, ну, эти... в малиновых пиджаках.',
    'new_russians_joke': 'Встречаются два новых русских. Один говорит: Я вчера Запорожец купил! Ну и как? Крылья бабочки, малиновый цвет, всё дела!',
    # Чапаев
    'chapaev_intro': 'Ну, Чапаев, понятное дело... он вообще к нам из гражданской войны пришёл, но мы его любим.',
    'chapaev_joke': 'Чапаев спрашивает Петьку: Петька, рубль есть? Нет, Василий Иваныч. А два есть? Тоже нет. Ну вот, а ты говоришь — капиталистическая революция.',
    # Сценарные фразы кофе
    'coffee_intro': 'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'coffee_obstacle': 'Бля, тут кто-то стоит... дай пройти.',
    'coffee_no_cup': 'Не вижу никакой чашки, Олег. Где кофе-то?',
    'coffee_got_it': 'Взял. Сейчас принесу.',
    'coffee_dropped': 'Ой... выронил, бля.',
    'coffee_done': 'Вот ваш кофе, Олег. Не обожгись, бля.',
}

os.makedirs('/content/presets', exist_ok=True)
preset_results = []

print('Генерация пресетов...')
for name, text in PRESETS.items():
    out = f'/content/presets/{name}.wav'
    print(f'  → {name} ({len(text)} символов)')
    try:
        t0 = time.time()
        wav, sr, _ = tts.infer(
            ref_file=REF_AUDIO,
            ref_text=REF_TEXT,
            gen_text=text,
            file_wave=out,
        )
        dt = time.time() - t0
        audio, sr_out = sf.read(out)
        dur = len(audio) / sr_out
        print(f'    ✅ {dur:.1f}сек, синтез {dt:.1f}с')
        preset_results.append({'name': name, 'duration': dur, 'synth_time': dt, 'path': out, 'text': text})
    except Exception as e:
        print(f'    ❌ Ошибка: {e}')

print(f'\n✅ Сгенерировано {len(preset_results)}/{len(PRESETS)} пресетов')

## 12. Прослушать пресеты

In [ ]:
for r in preset_results:
    print(f'\n>>> {r["name"]} ({r["duration"]:.1f}сек)')
    print(f'    {r["text"]}')
    display(Audio(r['path']))

## 13. Сохранить метаданные пресетов

JSON-манифест с информацией о каждом пресете. Нужен для `preset_player.py` на G1.

In [ ]:
import json

manifest_out = {
    'reference_audio': 'burunov_ref.wav',
    'reference_text': REF_TEXT,
    'reference_duration': ref['duration'],
    'model': 'hotstone228/F5-TTS-Russian',
    'device': 'colab-T4',
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'presets': [
        {
            'name': r['name'],
            'file': f"{r['name']}.wav",
            'duration_s': r['duration'],
            'text': r['text'],
        }
        for r in preset_results
    ],
}

with open('/content/presets/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_out, f, ensure_ascii=False, indent=2)

print('✅ /content/presets/manifest.json сохранён')
print(json.dumps(manifest_out, ensure_ascii=False, indent=2)[:1000])

## 14. Архивировать всё для переноса на G1

Создаём 2 архива:
- `f5_tts_russian.tar.gz` — модель (если понадобится деплоить локально на G1)
- `burunov_presets.tar.gz` — пресеты анекдотов + референс + манифест

In [ ]:
# Архив пресетов (маленький, ~5-15 МБ)
!cd /content && cp burunov_ref.wav presets/ && cp burunov_ref.txt presets/
!cd /content && tar -czf burunov_presets.tar.gz presets/
!ls -lh /content/burunov_presets.tar.gz

# Архив модели (большой, ~300-500 МБ)
print(f'\nМодель: {model_path}')
!cd /content && tar -czf f5_tts_russian.tar.gz -C $(dirname {model_path}) $(basename {model_path})
!ls -lh /content/f5_tts_russian.tar.gz

print('\n✅ Архивы готовы в /content/')
print('   burunov_presets.tar.gz — скачать через панель файлов слева')
print('   f5_tts_russian.tar.gz — скачать если хочешь деплоить модель на G1 локально')

## 15. Итоговый чеклист перед деплоем на G1

После прогона этого ноутбука у тебя должны быть:
- [ ] `burunov_presets.tar.gz` — со всеми wav анекдотов и реплик кофе
- [ ] `burunov_ref.wav` + `burunov_ref.txt` — референс для F5-TTS (если F5 пойдёт на G1)
- [ ] `f5_tts_russian.tar.gz` — модель (опц., если будете деплоить локально)
- [ ] Записан средний RTF (для решения F5 vs Piper)

### Дальше на G1:
```bash
# 1. Залить пресеты на G1
scp burunov_presets.tar.gz unitree@G1_IP:~/burunov-joke-bot/
ssh unitree@G1_IP
cd ~/burunov-joke-bot
mkdir -p data/preset_wav
tar -xzf burunov_presets.tar.gz -C data/preset_wav --strip-components=1

# 2. Если F5-TTS (RTF был < 1):
#    установить f5-tts через pip, скопировать модель, запустить f5_tts_server.py

# 3. Если Piper (RTF был > 2):
#    запустить edge_tts_server.py (старый код в репо)
```

## 16. (Опционально) Сравнить с другим референсом

Если результат не нравится — попробуй другой референс из топ-10 кандидатов.
Поменяй `CHOSEN_REF_INDEX_2` и прогони сравнение.

In [ ]:
# Сравнение 2-3 референсов на одной тестовой фразе
COMPARISON_TEXT = 'Угу, щас, Олег Тарасыч... кофеварку найду.'
REFS_TO_COMPARE = [0, 1, 2]  # индексы из candidates[:5]

os.makedirs('/content/ref_comparison', exist_ok=True)

for idx in REFS_TO_COMPARE:
    if idx >= len(candidates):
        continue
    ref_alt = candidates[idx]
    out = f'/content/ref_comparison/ref{idx}_{os.path.basename(ref_alt["audio_path"])}'
    print(f'\nРеференс #{idx}: {ref_alt["text"][:80]}')
    try:
        wav, sr, _ = tts.infer(
            ref_file=ref_alt['audio_path'],
            ref_text=ref_alt['text'],
            gen_text=COMPARISON_TEXT,
            file_wave=out,
        )
        display(Audio(out))
    except Exception as e:
        print(f'  ❌ {e}')

## 17. (Опционально) Fine-tune F5-TTS на полном датасете

Если zero-shot не устраивает — можно дообучить F5-TTS на всех 840 сегментах.
Это займёт 4-6 часов на T4. Раскомментируй код ниже если реально надо.

**Но скорее всего zero-shot с хорошим референсом хватит для демо.**

In [ ]:
# Подготовка датасета в формате F5-TTS для fine-tune
# import shutil
# os.makedirs('/content/finetune/audio', exist_ok=True)
# os.makedirs('/content/finetune/text', exist_ok=True)
#
# with open('/content/finetune/metadata.csv', 'w') as f:
#     for m in manifest:
#         text = m.get('text_corrected') or m.get('text', '')
#         if not text or len(text) < 10:
#             continue
#         src = os.path.join(DATASET_DIR, m['audio_path'])
#         name = os.path.basename(m['audio_path'])
#         shutil.copy(src, f'/content/finetune/audio/{name}')
#         f.write(f'{name}|{text}\n')
#
# print('Датасет подготовлен для fine-tune в /content/finetune/')
# print('Дальше — смотреть документацию F5-TTS по fine-tuning:')
# https://github.com/SWivid/F5-TTS/blob/main/docs/Finetune.md